# Airbnb market EDA, ETL and Visualization
#### https://www.kaggle.com/datasets/jasonairroi/airbnb-market-data-europe

# Analyse:
- What metrics are more relevant
- What amenities are more relevant
- Estimate how well an announcement will go
- Estimate price based on announcement
- Estimate which announcements are better for each season
- Analyse which cities are more visited

In [ ]:
# Importing libraries

import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns

In [21]:
pd.describe_option()

compute.use_bottleneck : bool
    Use the bottleneck library to accelerate if it is installed,
    the default is True
    Valid values: False,True
    [default: True] [currently: True]
compute.use_numba : bool
    Use the numba engine option for select operations if it is installed,
    the default is False
    Valid values: False,True
    [default: False] [currently: False]
compute.use_numexpr : bool
    Use the numexpr library to accelerate computation if it is installed,
    the default is True
    Valid values: False,True
    [default: True] [currently: True]
display.chop_threshold : float or None
    if set to a float value, all float values smaller than the given threshold
    will be displayed as exactly 0 by repr and friends.
    [default: None] [currently: None]
display.colheader_justify : 'left'/'right'
    Controls the justification of column headers. used by DataFrameFormatter.
    [default: right] [currently: right]
display.date_dayfirst : boolean
    When True, prints an

In [40]:
# Setting up libraries configurations

# Pandas:
pd.set_option('display.max_rows', 61)
pd.set_option('display.max_row', 61)
pd.set_option('display.date_yearfirst', True)
pd.option_context('display.max_rows', None)

# Matplotlib defaults


# Listings:

In [43]:
# Constants
LISTING_DATASET = "../data/listings.parquet"
LISTING_DATASET_CSV = "../data/listings.csv"
EXACT_COLUMNS_LISTINGS = ['listing_id', 'listing_type', 'room_type', 'cover_photo_url', 'photos_count', 'host_id', 
                'superhost', 'latitude', 'longitude', 'guests', 'bedrooms', 'beds', 'baths', 'registration', 
                'amenities', 'instant_book', 'professional_management', 'min_nights', 'cancellation_policy', 
                'currency', 'cleaning_fee', 'extra_guest_fee', 'num_reviews', 'rating_overall', 'rating_accuracy', 
                'rating_checkin', 'rating_cleanliness', 'rating_communication', 'rating_location', 'rating_value', 
                'ttm_revenue', 'ttm_revenue_native', 'ttm_avg_rate', 'ttm_avg_rate_native', 'ttm_occupancy', 
                'ttm_adjusted_occupancy', 'ttm_revpar', 'ttm_revpar_native', 'ttm_adjusted_revpar',
                'ttm_adjusted_revpar_native', 'ttm_reserved_days', 'ttm_blocked_days', 'ttm_available_days', 
                'ttm_total_days', 'l90d_revenue', 'l90d_revenue_native', 'l90d_avg_rate', 'l90d_avg_rate_native', 
                'l90d_occupancy', 'l90d_adjusted_occupancy', 'l90d_revpar', 'l90d_revpar_native', 
                'l90d_adjusted_revpar', 'l90d_adjusted_revpar_native', 'l90d_reserved_days', 'l90d_blocked_days', 
                'l90d_available_days', 'l90d_total_days', 'country', 'state', 'city'
]
#ACTUAL_COLUMNS_DTYPES = [ int,string,string,string,string,bool,float,float,int,int,int,int,bool,string,,false,7,Firm,TRY,0,0,8,5.0,5.0,5.0,4.9,5.0,4.9,5.0,24455.0,959613.0,390.1,15081.4,0.162,0.321,67.0,2629.1,132.9,5215.3,59,181,306,365,0.0,0.0,397.7,15607.0,0.0,0.0,0.0,0.0,0.0,0.0,0,90,90,90,Turkey,Muğla,Bodrum]
TARGET_COLS = ['ttm_revenue']

In [44]:
# Reading data
df_pqt = pd.read_parquet(LISTING_DATASET)
# df_dsv = pd.read_csv(LISTING_DATASET_CSV, sep=',')
df_pqt.head()

,listing_id,listing_type,room_type,cover_photo_url,photos_count,host_id,superhost,latitude,longitude,guests,...,l90d_revpar_native,l90d_adjusted_revpar,l90d_adjusted_revpar_native,l90d_reserved_days,l90d_blocked_days,l90d_available_days,l90d_total_days,country,state,city
0,121902,Entire home,entire_home,https://a0.muscache.com/im/pictures/77c0e3a9-0...,77,fe453949b595,false,37.0758,27.2426,6,...,0.0,0.0,0.0,0,90,90,90,Turkey,Muğla,Bodrum
1,805342,Entire condo,entire_home,https://a0.muscache.com/im/pictures/11494599/4...,16,59711ec4c245,false,37.0092,27.2563,3,...,0.0,0.0,0.0,0,0,90,90,Turkey,Muğla,Bodrum
2,805361,Entire home,entire_home,https://a0.muscache.com/im/pictures/bda48dbc-d...,34,d217bf6e3427,false,37.0292,27.4410,5,...,0.0,0.0,0.0,0,0,90,90,Turkey,Muğla,Bodrum
3,853827,Entire villa,entire_home,https://a0.muscache.com/im/pictures/26626113/a...,70,605fc7d80e02,false,37.0434,27.2517,16,...,536.3,0.0,0.0,2,0,88,90,Turkey,Muğla,Bodrum
4,967193,Entire villa,entire_home,https://a0.muscache.com/im/pictures/68107669/2...,37,3b963e8cd040,false,37.0429,27.3898,4,...,0.0,0.0,0.0,0,0,90,90,Turkey,Muğla,Bodrum


In [7]:
# Analysing columns
print(list(df_pqt.columns) == EXACT_COLUMNS_LISTINGS)

True


In [8]:
# Checking Non-Nulls, Nulls and Columns types
non_null_count_listings = df_pqt.notnull().sum().values
nulls = df_pqt.isna().sum()
types = df_pqt.dtypes.values

info_df = pd.DataFrame({
    'Non-Null Count': non_null_count_listings,
    'Null Count': nulls,
    'Dtypes': types
})
print(info_df)


                             Non-Null Count  Null Count Dtypes
listing_id                            95898           0    str
listing_type                          95329         569    str
room_type                             95329         569    str
cover_photo_url                       95102         796    str
photos_count                          95102         796    str
host_id                               95329         569    str
superhost                             95322         576    str
latitude                              95327         571    str
longitude                             95273         625    str
guests                                84865       11033    str
bedrooms                              78576       17322    str
beds                                  94336        1562    str
baths                                 94994         904    str
registration                          95064         834    str
amenities                             94960         938

In [ ]:
# Analysing real columns values (limited output size)
for col in df_pqt.columns:
    uniques = [i for i in df_pqt[col].unique()[:10]]
    print(col, ": ", uniques)

# One by one for better insight
#print(df_pqt['listing_id '].value_counts())
for value in df_pqt['listing_id '].unique():
    print(value)

In [115]:
# Defining correct columns dtypes

str_cols = [
    'listing_type', 'room_type', 'cover_photo_url', 'amenities', 'cancellation_policy', 
    'currency', 'country', 'state', 'city'
]

bool_cols = [
    'superhost', 'latitude', 'longitude', 'registration', 'instant_book', 'professional_management'

]

float_cols = [
    'beds', 'baths', 'rating_overall', 'rating_accuracy', 'rating_checkin', 'rating_cleanliness',
    'rating_communication', 'rating_location', 'rating_value', 'ttm_avg_rate', 'ttm_avg_rate_native',
    'ttm_occupancy', 'ttm_adjusted_occupancy', 'ttm_revpar', 'ttm_revpar_native', 'ttm_adjusted_revpar', 
    'ttm_adjusted_revpar_native', 'l90d_avg_rate', 'l90d_avg_rate_native', 'l90d_occupancy',
    'l90d_adjusted_occupancy', 'l90d_revpar', 'l90d_revpar_native', 'l90d_adjusted_revpar', 
    'l90d_adjusted_revpar_native'
]

int_cols = [
    'listing_id', 'photos_count', 'host_id', 'guests', 'bedrooms', 'min_nights', 
    'cleaning_fee', 'extra_guest_fee', 'num_reviews', 'ttm_revenue', 'ttm_revenue_native',
    'ttm_reserved_days', 'ttm_blocked_days', 'ttm_available_days', 'ttm_total_days',
    'l90d_revenue', 'l90d_revenue_native', 'l90d_reserved_days', 'l90d_blocked_days',
    'l90d_available_days', 'l90d_total_days'
]

print(len(EXACT_COLUMNS_LISTINGS) == (len(str_cols) + len(bool_cols) + len(float_cols) + len(int_cols)))

True


In [ ]:
# Changing columns to correct dtypes and removing outliers





